# 09 — Shared PySR Reproduction for Figures 2 and 3

**Question:** Can one symbolic-regression configuration recover plausible analytic $D_M/r_d(z)$ relations from DESI DR2 + Pantheon+ and DESI DR2 + DESY5?

This experiment evaluates both DESI error conventions: the public release covariance and the diagonal covariance constructed from the six rounded Table I standard deviations. The latter is a sensitivity comparison, not the authors' unpublished input matrix.

Each notebook execution creates a unique directory under `results/09_fig23_pysr_reproduction/`. Within that experiment, each of the four dataset × error runs has its own subfolder containing PySR's raw outputs, candidate equations, selected equation, diagnostics, a numerical fit benchmark, and a plot. Subsequent executions never overwrite previous experiment folders.

Scope: central distance reconstruction only. Notebook 10 will test random-seed stability; Notebook 11 will propagate uncertainty and reconstruct $w(z)$.

## 1. Imports and source data

Notebook 08 supplies $z$, $D_M/r_d$, covariance, row indices, and source metadata. The loader validates the array contract on every read.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "configs/pysr_fig23.yaml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loading import load_standard_dataset, with_desi_error_version
from src.symbolic_regression import (load_pysr_config, fit_exponential_reference,
    fit_symbolic_distance, export_fit_result, evaluate_sympy, chi_square)
CONFIG = load_pysr_config(PROJECT_ROOT / "configs/pysr_fig23.yaml")
SCENARIOS = ("fig2_pantheon", "fig3_desy5")
PREPARED_DIR = PROJECT_ROOT / "results/08_fig23_data_preparation"
DATA = {name: load_standard_dataset(PREPARED_DIR / f"{name}.npz") for name in SCENARIOS}
PAPER_EXPRESSIONS = {"fig2_pantheon": "33.3*z*exp(-0.274*z)",
                     "fig3_desy5": "30.1*z*exp(-0.262*z)"}
print("Scenarios:", SCENARIOS, "DESI error versions:", CONFIG["reproducibility"]["error_versions"])


Scenarios: ('fig2_pantheon', 'fig3_desy5') DESI error versions: ['public_release', 'paper_table']


## 2. Validate both error conventions before fitting

Only the six DESI diagonal entries change between versions. All measured central values and the full SN covariance stay identical. Both scenarios use the same six DESI BAO measurements. The paper Table I errors are rounded and may differ from those in the release.

In [2]:
VARIANTS = {(name, version): with_desi_error_version(DATA[name], version)
            for name in SCENARIOS for version in CONFIG["reproducibility"]["error_versions"]}
for name in SCENARIOS:
    public = VARIANTS[name, "public_release"]
    paper = VARIANTS[name, "paper_table"]
    np.testing.assert_array_equal(public["z"], paper["z"])
    np.testing.assert_array_equal(public["y"], paper["y"])
    np.testing.assert_array_equal(public["covariance"][6:,6:], paper["covariance"][6:,6:])
    assert np.count_nonzero(public["covariance"][:6,6:]) == 0
    assert np.count_nonzero(paper["covariance"][:6,6:]) == 0
    assert np.count_nonzero(paper["covariance"][:6,:6]
                            - np.diag(np.diag(paper["covariance"][:6,:6]))) == 0
np.testing.assert_array_equal(DATA["fig2_pantheon"]["y"][:6], DATA["fig3_desy5"]["y"][:6])
display(pd.DataFrame([{"scenario":name, "error_version":version, "n":len(data["y"]),
                       "BAO_sigma":np.array2string(data["sigma"][:6], precision=5)}
                      for (name, version), data in VARIANTS.items()]))


,scenario,error_version,n,BAO_sigma
0,fig2_pantheon,public_release,1596,[0.16837 0.17993 0.16178 0.32456 0.76356 0.53168]
1,fig2_pantheon,paper_table,1596,[0.167 0.177 0.152 0.318 0.76 0.531]
2,fig3_desy5,public_release,1760,[0.16837 0.17993 0.16178 0.32456 0.76356 0.53168]
3,fig3_desy5,paper_table,1760,[0.167 0.177 0.152 0.318 0.76 0.531]


## 3. Create a new experiment directory and record the configuration

The run ID includes UTC time and a random suffix to prevent collisions. The paper reports PySR operators, covariance-weighted loss, and `best` selection, but not its full search budget or seed. Values in `configs/pysr_fig23.yaml` are therefore independent reproduction choices. One fixed seed is used across the four comparisons.

The Julia loss evaluates $r^T C^{-1}r/n$ by prewhitening the residual with the inverse Cholesky factor. Each run saves that factor as a binary file so the exact objective can be audited without injecting millions of matrix values into Julia source. For large SN samples, full-covariance evaluation can be slow.

In [3]:
STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
EXPERIMENT_ID = f"fig23_fullcov_seed{CONFIG['reproducibility']['seed']}_{STAMP}_{uuid4().hex[:8]}"
EXPERIMENT_DIR = PROJECT_ROOT / CONFIG["output"]["root"] / EXPERIMENT_ID
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
experiment_record = {"experiment_id":EXPERIMENT_ID, "created_utc":datetime.now(timezone.utc).isoformat(),
                     "config":CONFIG, "scenario_ids":list(SCENARIOS),
                     "error_versions":list(CONFIG["reproducibility"]["error_versions"]),
                     "paper_equations":PAPER_EXPRESSIONS,
                     "status":"started"}
(EXPERIMENT_DIR / "experiment.json").write_text(json.dumps(experiment_record, indent=2)+"\n")
print("New experiment:", EXPERIMENT_DIR)


New experiment: /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec


## 4. Fit the paper's functional family as a numerical benchmark

This two-parameter generalized least-squares fit finds the best $a z\exp(bz)$ **within that family** for each input covariance. It helps distinguish coefficient differences caused by data choices from differences caused by PySR's search and model selection. This benchmark is not a symbolic-regression result.

In [4]:
benchmark_rows = []
for (name, version), data in VARIANTS.items():
    benchmark = fit_exponential_reference(data)
    paper_grid_at_data = evaluate_sympy(PAPER_EXPRESSIONS[name], data["z"])
    benchmark_rows.append({"scenario":name, "desi_error_version":version, **benchmark,
                           "paper_expression_loss":chi_square(data["y"], paper_grid_at_data,
                                                             data["covariance"])/len(data["y"])})
BENCHMARK = pd.DataFrame(benchmark_rows)
BENCHMARK.to_csv(EXPERIMENT_DIR / "fixed_family_benchmark.csv", index=False)
display(BENCHMARK)


,scenario,desi_error_version,a,b,chi2,loss,success,message,paper_expression_loss
0,fig2_pantheon,public_release,29.321837,-0.250232,1410.305845,0.883650,True,`ftol` termination condition is satisfied.,4.729393
1,fig2_pantheon,paper_table,29.322547,-0.250385,1410.494815,0.883769,True,`ftol` termination condition is satisfied.,4.754840
2,fig3_desy5,public_release,29.438635,-0.258654,1649.430406,0.937176,True,`ftol` termination condition is satisfied.,1.016472
3,fig3_desy5,paper_table,29.438778,-0.258650,1649.568894,0.937255,True,`ftol` termination condition is satisfied.,1.017151


## 5. Run the same covariance-aware PySR search on all four variants

The notebook writes each completed run immediately. If execution stops, completed run folders remain intact; rerunning the notebook creates a new experiment rather than overwriting them. The selected expression is not forced to pass through the origin. Its $D_M(0)$ value and finite/positive behavior are reported as diagnostics.

In [5]:
run_rows = []
for (name, version), data in VARIANTS.items():
    run_directory = EXPERIMENT_DIR / name / version
    print(f"Fitting {name} / {version} → {run_directory}", flush=True)
    result = fit_symbolic_distance(data, CONFIG, CONFIG["reproducibility"]["seed"],
                                   run_directory)
    paper_grid = evaluate_sympy(PAPER_EXPRESSIONS[name], result["z_grid"])
    benchmark = BENCHMARK.loc[(BENCHMARK.scenario == name)
                              & (BENCHMARK.desi_error_version == version)].iloc[0].to_dict()
    export_fit_result(result, run_directory, CONFIG, paper_grid, benchmark,
                      data["metadata"])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(data["z"][6:], data["y"][6:], yerr=data["sigma"][6:],
                fmt=".", ms=2, alpha=.22, elinewidth=.4, label="SN")
    ax.errorbar(data["z"][:6], data["y"][:6], yerr=data["sigma"][:6],
                fmt="o", color="black", ms=4, capsize=2, label="DESI DR2")
    ax.plot(result["z_grid"], result["selected_grid"], color="forestgreen",
            label="Selected PySR")
    ax.plot(result["z_grid"], paper_grid, "--", color="firebrick",
            label="Paper expression")
    ax.set(xlabel="Redshift z", ylabel=r"$D_M/r_d$",
           title=f"{name} · {version} · seed {CONFIG['reproducibility']['seed']}")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(run_directory / "distance_fit.png", dpi=170)
    plt.close(fig)
    row = result["summary"]
    row.update({"paper_expression":PAPER_EXPRESSIONS[name],
                "paper_expression_loss":benchmark["paper_expression_loss"],
                "benchmark_a":benchmark["a"], "benchmark_b":benchmark["b"],
                "benchmark_loss":benchmark["loss"]})
    run_rows.append(row)
    pd.DataFrame(run_rows).to_csv(EXPERIMENT_DIR / "summary.csv", index=False)
    print(f"Selected: {row['sympy_expression']} | loss={row['python_loss']:.5f}", flush=True)


Fitting fig2_pantheon / public_release → /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec/fig2_pantheon/public_release


Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


Selected: z*(28.99729801612583 - 5.7144561834408325*z) | loss=0.92008


Fitting fig2_pantheon / paper_table → /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec/fig2_pantheon/paper_table


Selected: sqrt(z) - z**3 + 24.04330984189465*z | loss=1.33281


Fitting fig3_desy5 / public_release → /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec/fig3_desy5/public_release


Selected: ((z - 0.007819989359353992)/0.02477353781129412)**0.8487126698019752 | loss=1.18359


Fitting fig3_desy5 / paper_table → /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec/fig3_desy5/paper_table


Selected: z*(28.774998567146834 + z*(-5.686938505863477)) | loss=0.98217


## 6. Compare structures, fit quality, and physical diagnostics

A close coefficient match is meaningful only for the same functional family. PySR's `best` expression may differ from the lowest-loss equation, and a single seed cannot establish stability. The zero-redshift and positivity checks are numerical diagnostics; a global analytical proof would require further work.

In [6]:
SUMMARY = pd.read_csv(EXPERIMENT_DIR / "summary.csv")
display(SUMMARY[["scenario", "desi_error_version", "complexity",
                 "sympy_expression", "python_loss", "paper_expression_loss",
                 "benchmark_a", "benchmark_b", "finite_on_grid",
                 "positive_for_z_gt_zero", "dm_at_zero",
                 "satisfies_dm_zero_constraint", "elapsed_seconds"]])
assert len(SUMMARY) == len(VARIANTS)
assert np.isfinite(SUMMARY["python_loss"]).all()
assert np.isfinite(SUMMARY["paper_expression_loss"]).all()
experiment_record["status"] = "complete"
experiment_record["completed_utc"] = datetime.now(timezone.utc).isoformat()
(EXPERIMENT_DIR / "experiment.json").write_text(json.dumps(experiment_record, indent=2)+"\n")
print("Saved complete experiment:", EXPERIMENT_DIR)


,scenario,desi_error_version,complexity,sympy_expression,python_loss,paper_expression_loss,benchmark_a,benchmark_b,finite_on_grid,positive_for_z_gt_zero,dm_at_zero,satisfies_dm_zero_constraint,elapsed_seconds
0,fig2_pantheon,public_release,7,z*(28.99729801612583 - 5.7144561834408325*z),0.920080,4.729393,29.321837,-0.250232,True,True,0.0,True,25.712700
1,fig2_pantheon,paper_table,9,sqrt(z) - z**3 + 24.04330984189465*z,1.332813,4.754840,29.322547,-0.250385,True,True,0.0,True,19.068586
2,fig3_desy5,public_release,7,((z - 0.007819989359353992)/0.0247735378112941...,1.183588,1.016472,29.438635,-0.258654,False,False,NaN,False,25.964380
3,fig3_desy5,paper_table,7,z*(28.774998567146834 + z*(-5.686938505863477)),0.982169,1.017151,29.438778,-0.258650,True,True,0.0,True,25.092788


Saved complete experiment: /Users/kemingliu/Desktop/Dark Energy/results/09_fig23_pysr_reproduction/fig23_fullcov_seed20260917_20260917T154033192841Z_6ae2faec


## 7. Interpretation and handoff

Read `summary.csv` with the four run folders. Each run preserves the PySR raw hall of fame, the selected/candidate equations, the whitening factor, a distance grid, a comparison plot, and a configuration JSON with input metadata and source hashes.

**Interpretation hierarchy:** First compare the paper equation's loss with the best fixed-family $az\exp(bz)$ loss under each data version. Next inspect which expression family PySR selected, its independent Python loss calculation, and whether $D_M(0)=0$. Finally compare public and paper-table errors within each SN scenario.

Do not interpret agreement of a single seed as structural stability; Notebook 10 repeats the search across seeds. Do not interpret these central fits as $w(z)$ confidence bands; Notebook 11 performs that propagation. A selected equation violating $D_M(0)=0$ should be labeled as a numerical fit and examined before differentiating.